# C12-classical-models — Session 1: Logistic Regression as a Trained Linear Classifier

*One 90-minute session. We reuse affine scores and gradient descent, but derive the probability
model and binary cross-entropy from binary events from scratch.*


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, brier_score_loss

SEED = 20260804
ATOL = 1e-10
RTOL = 1e-8
rng = np.random.default_rng(SEED)


## 1. From an affine score to log-odds

For one row $x\in\mathbb R^D$, let the affine score be $z=x^Tw+b$, with
$w\in\mathbb R^D$ and scalar $b$. A binary target is $y\in\{0,1\}$. If
$p=P(y=1\mid x)$, then the **odds** are $p/(1-p)$. Logistic regression declares

$$\log\frac{p}{1-p}=z.$$

Exponentiating and solving gives $p=e^z/(1+e^z)=1/(1+e^{-z})$. Thus a linear function of
features models **log-odds**, not probability directly. At $z=0$, odds are one and $p=0.5$.
Changing feature $x_j$ by one unit multiplies the odds by $e^{w_j}$ while other features stay
fixed.

**Checkpoint 1A.** If $z=\log 3$, what are the odds and probability?

**Checkpoint 1B.** If $w_j=-\log2$, how do the odds change after increasing $x_j$ by one?


## 2. Sigmoid and stable evaluation

The **sigmoid** is $\sigma(z)=1/(1+e^{-z})$. It is increasing, satisfies
$\sigma(-z)=1-\sigma(z)$, and has derivative $\sigma'(z)=\sigma(z)(1-\sigma(z))$.
Naively evaluating $e^{-z}$ overflows for a very negative $z$. A stable branch is

$$\sigma(z)=\begin{cases}1/(1+e^{-z})&z\ge0,\\e^z/(1+e^z)&z<0.\end{cases}$$

For an array `z` of any shape, the output has the same shape and float dtype. The public
function contract rejects a value that cannot become a numeric array, an empty array, or any NaN
or infinity by raising `ValueError`; numerical stability is not permission to accept invalid data.

**Checkpoint 2A.** Why is the negative branch safe when $z=-1000$?

**Checkpoint 2B.** Give $\sigma(0)$ and $\sigma'(0)$.


In [ ]:
def _finite_float_array(value, name):
    try:
        array = np.asarray(value, dtype=np.float64)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must be a numeric array") from exc
    if array.size == 0:
        raise ValueError(f"{name} must be nonempty")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{name} must contain only finite values")
    return array


def stable_sigmoid(z):
    z = _finite_float_array(z, "z")
    out = np.empty_like(z)
    nonnegative = z >= 0
    out[nonnegative] = 1.0 / (1.0 + np.exp(-z[nonnegative]))
    ez = np.exp(z[~nonnegative])
    out[~nonnegative] = ez / (1.0 + ez)
    return out

probe = np.array([-1000.0, 0.0, 1000.0])
prob = stable_sigmoid(probe)
assert prob.shape == probe.shape and prob.dtype == np.float64
assert np.all(np.isfinite(prob))
assert np.isclose(prob[1], 0.5, atol=ATOL, rtol=RTOL)
print(prob)


## 3. Deriving binary cross-entropy from an observed event

For one binary observation, the model assigns probability $p$ when $y=1$ and $1-p$ when
$y=0$. Both cases combine as the Bernoulli likelihood $p^y(1-p)^{1-y}$. Maximizing a product
over rows is equivalent to minimizing its negative logarithm. The per-row loss is therefore

$$\ell(y,p)=-\big[y\log p+(1-y)\log(1-p)\big].$$

This is **binary cross-entropy (BCE)**, derived here directly. With $p=\sigma(z)$, stable
algebra gives $\ell(y,z)=\max(z,0)-yz+\log(1+e^{-|z|})$. The mean loss over $N$ rows is
$L=N^{-1}\sum_i\ell_i$. Mean and sum losses are different objectives: their gradients differ
by $N$. The executable contract applies the same numeric/nonempty/finite validation to `z,y`,
then requires identical shapes and labels containing only exact 0 or 1 values.

**Checkpoint 3A.** What loss results from $y=1,p=1/4$?

**Checkpoint 3B.** Why is clipping $p$ not the same mathematical objective as stable logit BCE?


In [ ]:
def mean_bce_from_logits(z, y):
    z = _finite_float_array(z, "z")
    y = _finite_float_array(y, "y")
    if z.shape != y.shape:
        raise ValueError("z and y must have the same shape")
    if not np.all((y == 0.0) | (y == 1.0)):
        raise ValueError("y must contain only 0 or 1")
    losses = np.maximum(z, 0.0) - y * z + np.log1p(np.exp(-np.abs(z)))
    return float(losses.mean())


assert np.isclose(mean_bce_from_logits(np.array([0.0]), np.array([1.0])),
                  np.log(2.0), atol=ATOL, rtol=RTOL)
assert np.isfinite(mean_bce_from_logits(np.array([-10000.0, 10000.0]),
                                        np.array([0.0, 1.0])))

bad_nonnumeric_p06 = ["not-a-number"]
bad_shape_p06 = (np.array([0.0, 1.0]), np.array([1.0]))
bad_empty_p06 = np.array([], dtype=np.float64)
bad_nonfinite_p06 = np.array([0.0, np.inf], dtype=np.float64)
bad_label_p06 = np.array([0.0, 0.25], dtype=np.float64)
invalid_calls_p06 = [
    (stable_sigmoid, (bad_nonnumeric_p06,)),
    (stable_sigmoid, (bad_empty_p06,)),
    (stable_sigmoid, (bad_nonfinite_p06,)),
    (mean_bce_from_logits, bad_shape_p06),
    (mean_bce_from_logits, (np.array([0.0, 1.0]), bad_label_p06)),
]
for function_p06, arguments_p06 in invalid_calls_p06:
    try:
        function_p06(*arguments_p06)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid p06-style input must raise ValueError")


## 4. Deriving the mean gradient and shape ledger

Differentiate one loss. Since $d\ell/dp=-y/p+(1-y)/(1-p)$ and
$dp/dz=p(1-p)$, cancellation yields $d\ell/dz=p-y$. For a batch
$X\in\mathbb R^{N\times D}$, $z=Xw+b\mathbf1$, $p=\sigma(z)$, so

$$\nabla_wL=\frac{X^T(p-y)}N\in\mathbb R^D,\qquad
\frac{\partial L}{\partial b}=\frac1N\sum_i(p_i-y_i).$$

This derivation uses the scalar chain rule and coordinate sums; it assumes no matrix-calculus
shortcut. The $1/N$ factor belongs exactly once because $L$ is a mean.

**Checkpoint 4A.** State the shapes of `X`, `w`, `z`, `p-y`, and `grad_w`.

**Checkpoint 4B.** If every label is 0 but every probability exceeds 0.5, what sign does
`grad_b` have?


In [ ]:
X_demo = np.array([[1.0, 2.0], [-1.0, 0.5], [0.25, -0.75]])
y_demo = np.array([1.0, 0.0, 1.0])
w_demo = np.array([0.2, -0.1])
b_demo = 0.05
p_demo = stable_sigmoid(X_demo @ w_demo + b_demo)
grad_w_demo = X_demo.T @ (p_demo - y_demo) / X_demo.shape[0]
grad_b_demo = float((p_demo - y_demo).mean())
assert grad_w_demo.shape == (2,)
print("p", p_demo, "grad_w", grad_w_demo, "grad_b", grad_b_demo)


## 5. Worked example: deterministic NumPy training

We train on fixed arrays, starting at zeros and taking exactly 400 full-batch steps with learning
rate $0.2$. The update is `w -= learning_rate * grad_w` and likewise for $b$. A credible
training certificate records initial/final loss, finite values, parameter movement, and named
predictions. It does not merely say that the loop ran.

**Checkpoint 5A.** Why must the initial parameters be copied before training when auditing
movement?

**Checkpoint 5B.** If loss rises sharply on the first several steps, which scalar should you
inspect first?


In [ ]:
X_train = np.array([[-2.0, -1.0], [-1.0, -1.5], [-0.5, 0.0],
                    [0.5, 0.2], [1.0, 1.5], [2.0, 1.0]], dtype=np.float64)
y_train = np.array([0, 0, 0, 1, 1, 1], dtype=np.float64)
w = np.zeros(2, dtype=np.float64)
b = 0.0
w_initial = w.copy()
loss_trace = []
for _ in range(400):
    z = X_train @ w + b
    p = stable_sigmoid(z)
    loss_trace.append(mean_bce_from_logits(z, y_train))
    w -= 0.2 * (X_train.T @ (p - y_train) / len(y_train))
    b -= 0.2 * float((p - y_train).mean())

train_probability = stable_sigmoid(X_train @ w + b)
train_prediction = (train_probability >= 0.5).astype(np.int64)
assert np.isfinite(loss_trace).all()
assert loss_trace[-1] < loss_trace[0]
assert np.linalg.norm(w - w_initial) > 0.1
assert np.array_equal(train_prediction, y_train.astype(np.int64))
print("loss", loss_trace[0], "->", loss_trace[-1], "| w", w, "| b", b)


## 6. `LogisticRegression`, thresholds, accuracy, and calibration

`sklearn.linear_model.LogisticRegression` fits a regularized logistic model. In a
`Pipeline(StandardScaler(), LogisticRegression(...))`, scaling is learned from training data
only. `predict_proba(X)[:, 1]` returns positive-class probabilities; `predict(X)` uses a class
decision rule, commonly the 0.5 probability threshold for binary classes ordered `[0,1]`.

Accuracy scores thresholded labels. The Brier score averages $(p-y)^2$ and evaluates probability
quality. A model can keep the same accuracy while its probabilities become worse calibrated.
Choose a threshold on validation data for a named cost or metric, then evaluate once on test data.

**Checkpoint 6A.** Which method returns probabilities rather than class labels?

**Checkpoint 6B.** Can equal accuracy imply equal calibration? Explain.


In [ ]:
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, solver="lbfgs", random_state=SEED, max_iter=1000),
)
pipeline.fit(X_train, y_train.astype(np.int64))
sk_probability = pipeline.predict_proba(X_train)[:, 1]
sk_prediction = pipeline.predict(X_train)
print("accuracy", accuracy_score(y_train, sk_prediction),
      "| Brier", brier_score_loss(y_train, sk_probability))
assert sk_probability.shape == (6,)
assert np.all((sk_probability >= 0.0) & (sk_probability <= 1.0))


## 7. Separation, pitfalls, exam connections, and forward link

If a direction classifies every training point perfectly, unregularized BCE can keep decreasing
as $\|w\|$ grows; no finite coefficient need minimize it. This is **perfect separation**.
Regularization, an iteration cap, and held-out calibration checks make the behavior auditable.

**Common pitfalls.** A missing intercept shifts the boundary; naive sigmoid/BCE overflows; a
missing $1/N$ silently changes the learning rate; scaling before splitting leaks validation
information; `predict` is confused with `predict_proba`; and 0.5 is treated as universally
optimal despite unequal costs.

**Exam connection.** Expect exact odds or gradient arithmetic, a stable-function contract, or a
short training audit requiring both movement and independent behavior.

**Going deeper.** Session 2 replaces probability likelihood with margin geometry. The comparison
starts with output semantics, objective, scaling, probability/calibration, boundary geometry,
interpretability, nonlinear capacity, and validation method.

**Checkpoint 7A.** Under separation, why is “training accuracy is 100%” not a convergence proof?

**Checkpoint 7B.** Name two leakage-safe checks for probability predictions.


## Checkpoint answers

**1A.** Odds 3 and probability $3/4$. **1B.** They are multiplied by $1/2$.

**2A.** It evaluates $e^{-1000}$, which underflows harmlessly toward zero, rather than
$e^{1000}$. **2B.** $1/2$ and $1/4$.

**3A.** $-\log(1/4)=\log4$. **3B.** Clipping changes the value and derivative at the clip bounds;
stable logit algebra preserves the stated objective.

**4A.** $(N,D),(D),(N),(N),(D)$. **4B.** Positive.

**5A.** An alias would mutate with the live array and erase the baseline. **5B.** Learning rate.

**6A.** `predict_proba`. **6B.** No; thresholded decisions can match while probability distances
from the labels differ.

**7A.** Coefficients can keep growing while BCE falls, so accuracy does not certify a finite
optimum. **7B.** Fit preprocessing on training only and assess Brier/calibration on held-out data.
